# Projeto (Seção 6) — solução comentada

Continua a solução comentada da seção 5. Enunciado em [`06_projeto`](06_projeto.ipynb).

## 1. Solução

In [1]:
from datetime import date
from enum import Enum
from uuid import UUID, uuid4
from pydantic import UUID4, BaseModel, ConfigDict, Field, ValidationError, field_serializer
from pydantic.alias_generators import to_camel

MIN_MANUFACTURED_DATE = date(1980, 1, 1)


class AutomobileType(Enum):
    sedan = "Sedan"
    coupe = "Coupe"
    convertible = "Convertible"
    suv = "SUV"
    truck = "Truck"


def make_alias(field_name: str) -> str:
    return to_camel(field_name).removesuffix("_")


class Automobile(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        str_strip_whitespace=True,
        validate_default=True,
        validate_assignment=True,
        alias_generator=make_alias,
    )

    id_: UUID4 = Field(default_factory=uuid4)            # não anulável; novo uuid4 por instância
    manufacturer: str
    series_name: str
    type_: AutomobileType
    is_electric: bool = False
    manufactured_date: date = Field(validation_alias="completionDate", ge=MIN_MANUFACTURED_DATE)
    base_msrp_usd: float = Field(validation_alias="msrpUSD", serialization_alias="baseMSRPUSD")
    vin: str
    number_of_doors: int = Field(default=4, validation_alias="doors", ge=2, le=4, multiple_of=2)
    registration_country: str | None = None
    license_plate: str | None = None

    @field_serializer("manufactured_date", when_used="json-unless-none")
    def serialize_date(self, value: date) -> str:
        return value.strftime("%Y/%m/%d")

## 2. Testes do enunciado

In [2]:
data = {
    "id": "c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7",
    "manufacturer": "BMW",
    "seriesName": "M4",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

In [3]:
from uuid import UUID

expected_serialized_by_alias = {
    'id': UUID('c4e60f4a-3c7f-4da5-9b3f-07aee50b23e7'),
    'manufacturer': 'BMW',
    'seriesName': 'M4',
    'type': AutomobileType.convertible,
    'isElectric': False,
    'manufacturedDate': date(2023, 1, 1),
    'baseMSRPUSD': 93300.0,
    'vin': '1234567890',
    'numberOfDoors': 2,
    'registrationCountry': 'France',
    'licensePlate': 'AAA-BBB'
}

In [4]:
data_no_id = {
    "manufacturer": "BMW",
    "seriesName": "M4",
    "type": "Convertible",
    "isElectric": False,
    "completionDate": "2023-01-01",
    "msrpUSD": 93_300,
    "vin": "1234567890",
    "doors": 2,
    "registrationCountry": "France",
    "licensePlate": "AAA-BBB"
}

car = Automobile.model_validate(data)
assert car.model_dump(by_alias=True) == expected_serialized_by_alias

cars = [Automobile.model_validate(data_no_id) for _ in range(1000)]
ids = {c.id_ for c in cars}
assert all(isinstance(i, UUID) and i.version == 4 for i in ids)
assert len(ids) == len(cars)                          # 1000 instâncias, 1000 ids distintos
print(f"len(ids) para 1000 carros sem id: {len(ids)}")
print("testes do enunciado: ok")

len(ids) para 1000 carros sem id: 1000
testes do enunciado: ok


## 3. Testes extras — limites exatos

In [5]:
def errs(**changes) -> list[tuple]:
    try:
        Automobile.model_validate({**data, **changes})
    except ValidationError as ex:
        return [(err["loc"][0], err["type"]) for err in ex.errors()]
    return []

# fronteiras da data: 1980-01-01 é aceito (ge), 1979-12-31 não
assert errs(completionDate="1980-01-01") == []
print(f"completionDate='1979-12-31': {errs(completionDate='1979-12-31')}")

# portas: só 2 ou 4
for doors in (0, 1, 2, 3, 4, 5, 6):
    print(f"doors={doors}: {'ok' if not errs(doors=doors) else errs(doors=doors)}")

# id não anulável: None explícito agora é erro
print(f"id=None: {errs(id=None)}")

# a restrição também vale na atribuição (validate_assignment)
try:
    car.number_of_doors = 3
except ValidationError as ex:
    print(f"car.number_of_doors = 3: {ex.errors()[0]['type']}")
print("testes extras: ok")

completionDate='1979-12-31': [('completionDate', 'greater_than_equal')]
doors=0: [('doors', 'greater_than_equal')]
doors=1: [('doors', 'multiple_of')]
doors=2: ok
doors=3: [('doors', 'multiple_of')]
doors=4: ok
doors=5: [('doors', 'multiple_of')]
doors=6: [('doors', 'less_than_equal')]
id=None: [('id', 'uuid_type')]
car.number_of_doors = 3: multiple_of
testes extras: ok


## 4. Comparação com a solução do instrutor

| ponto | instrutor | esta solução |
|---|---|---|
| `id_` | `Field(alias="id", default_factory=uuid4)` | `Field(default_factory=uuid4)` — alias via `make_alias` |
| data mínima | `ge=date(1980, 1, 1)` inline | constante nomeada `MIN_MANUFACTURED_DATE` |
| portas | `ge=2, le=4, multiple_of=2` | idem |
| teste de unicidade | 2 instâncias | 1000 instâncias + versão do UUID |
| fronteiras | um caso inválido | limites exatos (1979-12-31 × 1980-01-01; 0 a 6 portas), `id=None`, atribuição |

> ⚠️ **Detalhe que vale conhecer:** `validate_default=True` faz o `uuid4()` da factory também passar pela validação `UUID4` — custo
> pequeno, mas real em modelos criados aos milhares. Aqui é proposital (garante que a factory devolve o tipo certo).